In [ ]:
### This is the same as the other feature extraction notebook, alebit with different masks

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
from skimage.measure import label, regionprops, regionprops_table
from skimage.transform import rotate
from sklearn.cluster import KMeans
import umap
import pickle
import csv
from skimage.filters import sobel
from skimage.segmentation import expand_labels, watershed
import skimage.segmentation as segmentation
from collections import defaultdict
from pathlib import Path
import multiprocessing
import time
import logging


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [PID %(process)d] %(message)s",
)

In [3]:
def get_binary_from_mask(mask):
    # Convert input to an 8-bit grayscale image
    core_img_grey = (mask // 256).astype(np.uint8)
    
    # Apply binary thresholding 
    _, binary = cv2.threshold(core_img_grey, 1, 1, cv2.THRESH_OTSU)
    
    # Find external contours
    #contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    return binary

In [5]:
output_dir = "path_with_masks"

# Map suffixes in filenames to keys in our nested dictionary
suffix_map = {
    "expanded_overlap": "_expanded_overlap.npy",
    "intensity_wo_DAPI": "_int_img_wo_DAPI.npy",  # matches your filenames
    "overlap_masks": "_overlap_mask.npy",
    "DAPI_masks": "_dapi_masks.npy",
    "cell_body_mask": "_cell_body_mask.npy",
}

results_dict = {}

for f in os.listdir(output_dir):
    if not f.endswith(".npy"):
        continue

    # Find which suffix this file matches
    for key, suffix in suffix_map.items():
        if f.endswith(suffix):
            # TMA name is the filename without the suffix
            tma_name = f[:-len(suffix)]
            if tma_name not in results_dict:
                results_dict[tma_name] = {k: "" for k in suffix_map.keys()}
            # Assign the file path
            results_dict[tma_name][key] = os.path.join(output_dir, f)
            break  # stop checking other suffixes


In [64]:
import time
import pandas as pd
from skimage.measure import label, regionprops_table

def region_props2(label_path, intensity_path, filename):
    """
    dapi_mask: 2D array of labeled nuclei (H, W)
    imgs: intensity image(s), shape (num_cycles, num_channels, H, W) or (H, W)
    filename: string
    """
    start_time = time.time()
    logging.info(f"Processing {filename}...")#
    
    intensity_path = np.load(intensity_path,  mmap_mode='r')
    raw_dapi = intensity_path[0,0,:,:]

    overlap_imgs = np.load(label_path)

    labels = label(overlap_imgs)
    
    all_props = []

    props = regionprops_table(
            labels,
            intensity_image= raw_dapi,
            properties=[
                'label', 'centroid', 'orientation',
                'axis_major_length', 'axis_minor_length',
                'area', 'eccentricity', 'perimeter', 'mean_intensity'
            ]
        )
    props_df = pd.DataFrame(props)
    props_df['filename'] = f"{filename}"
    all_props.append(props_df)

    # Combine all intensity measurements
    master_df = pd.concat(all_props, axis=0, ignore_index=True)

    elapsed = time.time() - start_time
    logging.info(f"Elapsed time for {filename}: {elapsed:.2f} seconds")
    return master_df


In [65]:
if __name__ == '__main__':
    # Prepare inputs
    inputs = [
        (value["overlap_masks"], value["raw_DAPI"], key)
        for key, value in results_dict.items()
    ]
    def wrapper(args):
        labels, imgs, filename = args
        return region_props2(labels, imgs, filename)

    # Use a multiprocessing Pool
    with multiprocessing.Pool(processes=64) as pool:
        dfs = pool.map(wrapper, inputs)
        
    # Combine results
    dapi_df = pd.concat(dfs, ignore_index=True)
    output_dir = "whatever_you_choose.csv" 
    dapi_df.to_csv(output_dir, index=False, na_rep='NaN')


2026-01-20 20:05:58,831 [PID 3834148] Processing TMA0_3...
2026-01-20 20:05:58,831 [PID 3834149] Processing TMA9_44...
2026-01-20 20:05:58,830 [PID 3834145] Processing TMA9_15...
2026-01-20 20:05:58,830 [PID 3834146] Processing TMA0_89...
2026-01-20 20:05:58,831 [PID 3834147] Processing TMA5_10...
2026-01-20 20:05:58,833 [PID 3834157] Processing TMA0_33...
2026-01-20 20:05:58,831 [PID 3834150] Processing TMA11_80...
2026-01-20 20:05:58,832 [PID 3834151] Processing TMA11_44...
2026-01-20 20:05:58,833 [PID 3834154] Processing TMA5_55...
2026-01-20 20:05:58,833 [PID 3834155] Processing TMA11_26...
2026-01-20 20:05:58,833 [PID 3834156] Processing TMA14_24...
2026-01-20 20:05:58,835 [PID 3834162] Processing TMA14_58...
2026-01-20 20:05:58,834 [PID 3834159] Processing TMA6_45...
2026-01-20 20:05:58,832 [PID 3834152] Processing TMA0_83...
2026-01-20 20:05:58,832 [PID 3834153] Processing TMA6_10...
2026-01-20 20:05:58,835 [PID 3834167] Processing TMA2_2...
2026-01-20 20:05:58,836 [PID 3834169]

         label   centroid-0   centroid-1  orientation  axis_major_length  \
3694088   4093  6420.269521  3536.279597     1.245370          26.039871   
3694089   4094  6422.359206  3229.826715     1.556905          36.339194   
3694090   4095  6433.809249  3293.481214     1.370447          52.331693   
3694091   4096  6442.582079  3435.906293     1.507306          62.354971   
3694092   4097  6442.367876  3524.834197     1.292771          21.057289   

         axis_minor_length    area  eccentricity   perimeter  mean_intensity  \
3694088          19.566478   397.0      0.659842   73.112698     6059.594458   
3694089          19.698293   554.0      0.840335   93.597980     3396.667870   
3694090          18.163587   692.0      0.937833  121.840620     4297.111272   
3694091          31.256197  1462.0      0.865295  167.781746     4611.719562   
3694092          11.727296   193.0      0.830564   52.627417     4590.590674   

         filename  
3694088  TMA10_35  
3694089  TMA10_35  
36